In [2]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix

In [3]:
caminho_pasta_tratado = '../../dataset tratado/cicids2017/'

nome_dados_treinamento = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos.csv'
nome_dados_teste       = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos.csv'

In [4]:
print("Carregando dataset de treinamento...")
df_treino_full = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
print(f"Dataset completo: {df_treino_full.shape}")

df_treino_benign = df_treino_full[df_treino_full['Label'] == 'BENIGN'].copy()
print(f"Apenas BENIGN: {df_treino_benign.shape}")

X_treino = df_treino_benign.drop('Label', axis=1).values

display(df_treino_benign.head())

Carregando dataset de treinamento...
Dataset completo: (1979513, 52)
Apenas BENIGN: (1590306, 52)


,Bwd Packet Length Mean,Average Packet Size,Packet Length Mean,Bwd Packet Length Std,Packet Length Variance,Init_Win_bytes_forward,Packet Length Std,Bwd Packet Length Max,Total Length of Fwd Packets,Avg Bwd Segment Size,...,Flow Duration,Fwd Packet Length Min,Down/Up Ratio,URG Flag Count,Bwd IAT Max,Active Mean,Bwd IAT Mean,Active Min,Bwd IAT Total,Label
0,0.001034,0.002825,0.002637,0.000000,1.219643e-05,0.005341,0.003493,0.000307,2.945736e-06,0.001034,...,3.229166e-05,0.000000,0.00000,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,BENIGN
1,0.000103,0.000096,0.000100,0.000109,2.232143e-08,0.018539,0.000149,0.000102,0.000000e+00,0.000103,...,2.272266e-03,0.000000,0.00641,1.0,7.645083e-04,0.0,5.603375e-04,0.0,2.241350e-03,BENIGN
2,0.000000,0.002312,0.001798,0.000000,0.000000e+00,0.078049,0.000000,0.000000,9.302326e-07,0.000000,...,1.333333e-07,0.002581,0.00000,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,BENIGN
3,0.010344,0.016182,0.015103,0.000000,3.428571e-06,0.000000,0.001852,0.003072,6.821705e-06,0.010344,...,1.233333e-06,0.018925,0.00641,0.0,3.333333e-08,0.0,3.333333e-08,0.0,3.333333e-08,BENIGN
4,0.019826,0.020355,0.018998,0.000000,9.905357e-05,0.000000,0.009955,0.005888,4.496124e-06,0.019826,...,1.449567e-03,0.012473,0.00641,0.0,2.500000e-08,0.0,2.500000e-08,0.0,2.500000e-08,BENIGN


In [5]:
# Arquitetura do Autoencoder
# Encoder: n → 32 → 16 → 8 | Decoder: 8 → 16 → 32 → n
n_features = X_treino.shape[1]

inputs = Input(shape=(n_features,))

# Encoder
encoded = Dense(32, activation='relu')(inputs)
encoded = Dense(16, activation='relu')(encoded)
encoded = Dense(8, activation='relu')(encoded)

# Decoder
decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(32, activation='relu')(decoded)
outputs = Dense(n_features, activation='sigmoid')(decoded)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 51)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 51)             │         1,683 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,699 (18.36 KB)

 Trainable params: 4,699 (18.36 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Treinamento do Autoencoder (apenas em tráfego BENIGN)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Iniciando treinamento do Autoencoder...")
inicio_treino = time.time()

history = autoencoder.fit(
    X_treino, X_treino,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

fim_treino = time.time()
print(f"\nTreinamento concluído! Tempo total: {fim_treino - inicio_treino:.2f} segundos.")

Iniciando treinamento do Autoencoder...
Epoch 1/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 901us/step - loss: 0.0069 - val_loss: 0.0033
Epoch 2/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 887us/step - loss: 0.0032 - val_loss: 0.0031
Epoch 3/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 870us/step - loss: 0.0030 - val_loss: 0.0030
Epoch 4/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 897us/step - loss: 0.0030 - val_loss: 0.0030
Epoch 5/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 877us/step - loss: 0.0030 - val_loss: 0.0030
Epoch 6/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 878us/step - loss: 0.0022 - val_loss: 7.2945e-04
Epoch 7/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 877us/step - loss: 7.2168e-04 - val_loss: 7.2060e-04
Epoch 8/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 881us/step - loss: 7.1101e-04 - val_loss: 7.0630e-04
Epoch 9/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 879us/step - loss: 7.0563e-04 - val_loss: 7.0170e-04
Epoch 10/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 890us/step - loss: 7.0214e-04 - val_loss: 6.9874e-04
Epoch 11/50
5591/5591 ━

In [7]:
# Definindo o limiar de anomalia (percentil 95 do erro de reconstrução no treino BENIGN)
X_treino_rec = autoencoder.predict(X_treino, verbose=0)
erros_treino = np.mean(np.power(X_treino - X_treino_rec, 2), axis=1)

PERCENTIL = 95
limiar = np.percentile(erros_treino, PERCENTIL)

print(f"Estatísticas do erro de reconstrução (BENIGN treino):")
print(f"  Média:   {erros_treino.mean():.6f}")
print(f"  Desvio:  {erros_treino.std():.6f}")
print(f"  Máximo:  {erros_treino.max():.6f}")
print(f"  Limiar ({PERCENTIL}º percentil): {limiar:.6f}")

Estatísticas do erro de reconstrução (BENIGN treino):
  Média:   0.000633
  Desvio:  0.003010
  Máximo:  0.066262
  Limiar (95º percentil): 0.004391


In [8]:
def avaliar_autoencoder(autoencoder, limiar, df_teste, nome_cenario, benign_label):
    X_teste = df_teste.drop('Label', axis=1).values
    y_teste_real = df_teste['Label'].values

    X_teste_rec = autoencoder.predict(X_teste, verbose=0)
    erros_teste = np.mean(np.power(X_teste - X_teste_rec, 2), axis=1)

    y_pred_binario = np.where(erros_teste > limiar, 'ATTACK', 'BENIGN')
    y_real_binario = np.where(y_teste_real == benign_label, 'BENIGN', 'ATTACK')

    labels_bin = ['BENIGN', 'ATTACK']
    cm = confusion_matrix(y_real_binario, y_pred_binario, labels=labels_bin)
    cm_df = pd.DataFrame(
        cm,
        index=[f"Real_{l}" for l in labels_bin],
        columns=[f"Pred_{l}" for l in labels_bin]
    )

    print(f"\n{'='*60}")
    print(f"CENÁRIO: {nome_cenario}")
    print(f"Total de amostras: {len(y_teste_real)}")
    print(f"  BENIGN: {(y_real_binario == 'BENIGN').sum()}")
    print(f"  ATTACK: {(y_real_binario == 'ATTACK').sum()}")
    print(f"\n=== MATRIZ DE CONFUSÃO ===")
    display(cm_df.style.format("{:.0f}"))
    print(f"\n=== RELATÓRIO DE MÉTRICAS ===")
    print(classification_report(y_real_binario, y_pred_binario, labels=labels_bin, zero_division=0))

    return y_real_binario, y_pred_binario, cm

In [9]:
CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'ATTACK': 'ATTACK'}


def escape_latex(value):
    replacements = {
        "\\": "\\textbackslash{}",
        "&": "\\&",
        "%": "\\%",
        "$": "\\$",
        "#": "\\#",
        "_": "\\_",
        "{": "\\{",
        "}": "\\}",
        "~": "\\textasciitilde{}",
        "^": "\\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return f"\\ok{{{value}}}"
    if value != 0:
        return f"\\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [escape_latex(CLASS_ALIASES_LATEX.get(l, l)) for l in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((f"Real\\_{escape_latex(CLASS_ALIASES_LATEX.get(real_label, real_label))}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        f"        \\begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        "            \\hline",
        format_row("", headers),
        "            \\hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        "            \\hline",
        "        \\end{tabular}",
        "    }",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values, y_pred_values,
        labels=class_labels, output_dict=True, zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            escape_latex(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        ["\\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        ["\\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        ["\\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + " \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        "    \\begin{tabular}{lrrrr}",
        "        \\hline",
        format_row(headers),
        "        \\hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        "        \\hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        "        \\hline",
        "    \\end{tabular}",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)

In [10]:
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)
y_real, y_pred, cm = avaliar_autoencoder(autoencoder, limiar, df_teste, 'Teste Completo', 'BENIGN')

labels_bin = ['BENIGN', 'ATTACK']

print(make_latex_confusion_matrix(
    cm, labels_bin,
    'Autoencoder — CICIDS2017 com MDI (Teste Completo) — Matriz de Confusão',
    'table:ae_cicids_mdi_completo_mc',
))
print()
print(make_latex_metrics_report(
    y_real, y_pred, labels_bin,
    'Autoencoder — CICIDS2017 com MDI (Teste Completo) — Relatório de Métricas',
    'table:ae_cicids_mdi_completo_metricas',
))


CENÁRIO: Teste Completo
Total de amostras: 848363
  BENIGN: 681014
  ATTACK: 167349

=== MATRIZ DE CONFUSÃO ===


,Pred_BENIGN,Pred_ATTACK
Real_BENIGN,646831,34183
Real_ATTACK,112014,55335



=== RELATÓRIO DE MÉTRICAS ===
              precision    recall  f1-score   support

      BENIGN       0.85      0.95      0.90    681014
      ATTACK       0.62      0.33      0.43    167349

    accuracy                           0.83    848363
   macro avg       0.74      0.64      0.66    848363
weighted avg       0.81      0.83      0.81    848363

\begin{table}[H]
    \centering
    \small
        \begin{tabular}{l|rr}
            \hline
                         & BENIGN       & ATTACK      \\
            \hline
            Real\_BENIGN & \ok{646831}  & \err{34183} \\
            Real\_ATTACK & \err{112014} & \ok{55335}  \\
            \hline
        \end{tabular}
    }
    \caption{Autoencoder — CICIDS2017 com MDI (Teste Completo) — Matriz de Confusão}
    \label{table:ae_cicids_mdi_completo_mc}
\end{table}

\begin{table}[H]
    \centering
    \small
    \begin{tabular}{lrrrr}
        \hline
        Classe                   & Precisão & Revocação & F1-score & Suporte \\
      